# Logistic regression baseline for occupancy detection

Predicts presence (0 = empty, 1 = occupied) in apartment 3 of the KTH Live-In Lab (May 2020)
from indoor sensor data, and compares five time resolutions (project step 4).

Works on Windows and macOS: paths are relative to this notebook's folder (`notebooks/`).

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, recall_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Settings

In [ ]:
DATA_DIR = Path("..") / "data" / "processed"
RESOLUTIONS = ["1min_master", "5min", "15min", "30min", "60min"]

TIME_COL = "timestamp"
FEATURES = ["co2_ppm", "rh_pct", "temp_in_c"]
TARGET = "presence"
AGE_COLS = ["co2_age_min", "rh_age_min", "temp_in_age_min", "temp_out_age_min", "light_age_min"]

# Same test period for every resolution, so the scores are comparable (last ~7 days).
TEST_START = "2020-05-25"


def load(resolution):
    path = DATA_DIR / f"lgh3_may2020_{resolution}.csv"
    return pd.read_csv(path, parse_dates=[TIME_COL]).sort_values(TIME_COL)

## A first look at the data (15-min resolution)

In [ ]:
df = load("15min")
print(f"{len(df)} rows, {df[TIME_COL].min()} -> {df[TIME_COL].max()}")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Share of time occupied: {df[TARGET].mean():.1%}")
df.head()

## Data quality: how old are the sensor readings?
The sensors do not report every minute; the last reading was carried forward and `*_age_min`
says how many minutes old it is. Large ages are sensor gaps to document in the report (step 3).
The age columns are **not** used as model inputs: sensors tend to report when something changes,
so the model could learn reporting patterns instead of occupancy.

In [ ]:
master = load("1min_master")
pd.DataFrame({
    "max age (min)": master[AGE_COLS].max(),
    "share > 30 min old": (master[AGE_COLS] > 30).mean(),
    "share > 60 min old": (master[AGE_COLS] > 60).mean(),
}).round(3)

## Train and evaluate one resolution
- Chronological split: train before `TEST_START`, test after (no shuffling).
- Tune `C` and `class_weight` with time-series cross-validation on the training period.
  The scaler is inside the pipeline, so it is fit on training folds only.
- Score with macro F1, because the apartment is occupied ~74% of the time: a model that always
  says "occupied" already gets ~74% accuracy (the baseline below).

In [ ]:
def evaluate(df):
    train = df[df[TIME_COL] < TEST_START]
    test = df[df[TIME_COL] >= TEST_START]
    X_train, y_train = train[FEATURES], train[TARGET].astype(int)
    X_test, y_test = test[FEATURES], test[TARGET].astype(int)

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    search = GridSearchCV(
        model,
        param_grid={
            "logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "logisticregression__class_weight": [None, "balanced"],
        },
        cv=TimeSeriesSplit(n_splits=5),
        scoring="f1_macro",
    )
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)

    scores = {
        "train rows": len(train),
        "test rows": len(test),
        "best C": search.best_params_["logisticregression__C"],
        "class_weight": search.best_params_["logisticregression__class_weight"],
        "CV macro F1": search.best_score_,
        "test macro F1": f1_score(y_test, y_pred, average="macro"),
        "test accuracy": accuracy_score(y_test, y_pred),
        "baseline accuracy": y_test.mean(),  # always predict "occupied"
        "recall empty": recall_score(y_test, y_pred, pos_label=0),
    }
    return search, scores, y_test, y_pred

## Compare time resolutions (step 4)

In [ ]:
runs = {}
for res in RESOLUTIONS:
    runs[res] = evaluate(load(res))

results = pd.DataFrame({res: run[1] for res, run in runs.items()}).T
results

## Best resolution in detail
Chosen by **cross-validation** score, not test score, so the test period stays untouched by model selection.

In [ ]:
best_res = results["CV macro F1"].astype(float).idxmax()
search, scores, y_test, y_pred = runs[best_res]
print(f"Best resolution: {best_res}\n")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["empty", "occupied"]))

## Coefficients (on standardized features)
Larger magnitude = stronger influence on the prediction; positive = pushes towards "occupied".

In [ ]:
coefs = search.best_estimator_.named_steps["logisticregression"].coef_[0]
pd.Series(coefs, index=FEATURES).sort_values()